# REMX volatility-model pipeline

Runs the three model-estimation notebooks in sequence and stitches their captured outputs into a single combined report at `REPORTS/garch_report.txt`.

1. `03-GP-models.ipynb` — GARCH(1,1) baseline
2. `04-extension-models.ipynb` — Sentiment-augmented EGARCH(1,1)-X (Gaussian)
3. `05-robustness_tests.ipynb` — EGARCH(1,1)-X with Student-t innovations + LR test


In [ ]:
import nbformat
from nbclient import NotebookClient

NOTEBOOKS = [
    '03-GP-models.ipynb',
    '04-extension-models.ipynb',
    '05-robustness_tests.ipynb',
]

for nb_path in NOTEBOOKS:
    print(f"Executing {nb_path} ...")
    nb = nbformat.read(nb_path, as_version=4)
    client = NotebookClient(nb, timeout=600, kernel_name='python3')
    client.execute()
    nbformat.write(nb, nb_path)
    print(f"  done.")


## Combined report

Reads each notebook's persisted capture and stitches them into `REPORTS/garch_report.txt` with one section per model.


In [ ]:
from pathlib import Path
from datetime import datetime

REPORTS_DIR = Path("REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)
OUT_PATH = REPORTS_DIR / "garch_report.txt"

sections = [
    ("GARCH(1,1) — Gaussian (arch_model)",         REPORTS_DIR / "_capture_garch.txt"),
    ("EGARCH(1,1)-X — Gaussian innovations",       REPORTS_DIR / "_capture_egarch_x_normal.txt"),
    ("EGARCH(1,1)-X — Student-t innovations",      REPORTS_DIR / "_capture_egarch_x_studentt.txt"),
]

def _banner(title, char='='):
    bar = char * 78
    return f"{bar}\n{title}\n{bar}\n"

with OUT_PATH.open("w") as f:
    f.write(_banner(f"REMX volatility-model estimation report  "
                    f"(generated {datetime.now():%Y-%m-%d %H:%M})"))
    f.write("\n")
    for title, cap_path in sections:
        block = _banner(title, '-')
        body  = cap_path.read_text() if cap_path.exists() else "(capture file not found - re-run the source notebook)\n"
        f.write(block)
        f.write(body)
        f.write("\n\n")
        print(block + body)

print(f"Saved -> {OUT_PATH}  ({OUT_PATH.stat().st_size:,} bytes)")
